# L2: Create Agents to Research and Write an Article

In this lesson, you will be introduced to the foundational concepts of multi-agent systems and get an overview of the crewAI framework.

In [ ]:
# !pip install --upgrade huggingface_hub

In [1]:
# !pip install crewai

In [9]:
# !pip install crewai==0.28.8 crewai_tools==0.1.6 langchain_community==0.0.29

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 3.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 7.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
INFO: pip is looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of embedchain to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of embedchain[github,youtube] to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of embedchain[github,youtube] to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.7/160.7 kB 11.8 MB/s et

In [3]:
import os
from google.colab import drive
drive.mount('content')

Mounted at content


In [4]:
os.getcwd()

'/content'

In [5]:
%cd content/MyDrive/Colab ""Notebooks/CrewAI_Course/Agentic_AI

/content/content/MyDrive/Colab Notebooks/CrewAI_Course/Agentic_AI


In [6]:
# Warning control
import warnings
warnings.filterwarnings('ignore')

In [10]:
# from crewai import Agent, Task, Crew , LLM
from  crewai  import Agent, Task, Crew, Process

As an LLM for the  agents, we'll be using OpenAI's gpt-3.5-turbo.

In [11]:
# !pip install utils

In [12]:
import google.generativeai as genai
from google.colab import userdata

In [13]:


OPENAI_API_KEY=userdata.get('My_API_Key')
os.environ["OPENAI_MODEL_NAME"] = 'gpt-3.5-turbo'

os.environ['OPENAI_API_KEY']=OPENAI_API_KEY


In [14]:
from langchain_community.llms import HuggingFaceHub

llm = HuggingFaceHub(
    repo_id="HuggingFaceH4/zephyr-7b-beta",
    huggingfacehub_api_token="<HF_TOKEN_HERE>",
    task="text-generation",
)

/usr/local/lib/python3.10/dist-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `HuggingFaceHub` was deprecated in LangChain 0.0.21 and will be removed in 0.2.0. Use HuggingFaceEndpoint instead.
  warn_deprecated(


In [15]:

HF_KEY=userdata.get('HF_Token')
os.environ["HF_Model"] = 'Zephyr-7b'

os.environ['HF_Key']=HF_KEY

### Agent: Planner

In [16]:
planner=Agent(role='Content Planner'
,goal='Plan engaging and factually accurate content on {topic}'
  ,  backstory="You're working on planning a blog article "
              "about the topic: {topic}."
              "You collect information that helps the "
              "audience learn something "
              "and make informed decisions. "
              "Your work is the basis for "
              "the Content Writer to write an article on this topic.",
    allow_delegation=False,
	verbose=True,
function_calling_llm= llm
)

### Agent: Writer


In [17]:


writer = Agent(
    role="Content Writer",
    goal="Write insightful and factually accurate "
         "opinion piece about the topic: {topic}",
    backstory="You're working on a writing "
              "a new opinion piece about the topic: {topic}. "
              "You base your writing on the work of "
              "the Content Planner, who provides an outline "
              "and relevant context about the topic. "
              "You follow the main objectives and "
              "direction of the outline, "
              "as provide by the Content Planner. "
              "You also provide objective and impartial insights "
              "and back them up with information "
              "provide by the Content Planner. "
              "You acknowledge in your opinion piece "
              "when your statements are opinions "
              "as opposed to objective statements.",
    allow_delegation=False,
    verbose=True,
    function_calling_llm= llm
)





### Agent:- Editor


In [18]:
editor = Agent(
    role="Editor",
    goal="Edit a given blog post to align with "
         "the writing style of the organization. ",
    backstory="You are an editor who receives a blog post "
              "from the Content Writer. "
              "Your goal is to review the blog post "
              "to ensure that it follows journalistic best practices,"
              "provides balanced viewpoints "
              "when providing opinions or assertions, "
              "and also avoids major controversial topics "
              "or opinions when possible.",
    allow_delegation=False,
    verbose=True
    ,function_calling_llm= llm
)

### Task - Plan

In [19]:
plan = Task(
    description=(
        "1. Prioritize the latest trends, key players, "
            "and noteworthy news on {topic}.\n"
        "2. Identify the target audience, considering "
            "their interests and pain points.\n"
        "3. Develop a detailed content outline including "
            "an introduction, key points, and a call to action.\n"
        "4. Include SEO keywords and relevant data or sources."
    ),
    expected_output="A comprehensive content plan document "
        "with an outline, audience analysis, "
        "SEO keywords, and resources.",
    agent=planner
)



### Task :- Write


In [20]:
write = Task(
    description=(
        "1. Use the content plan to craft a compelling "
            "blog post on {topic}.\n"
        "2. Incorporate SEO keywords naturally.\n"
		"3. Sections/Subtitles are properly named "
            "in an engaging manner.\n"
        "4. Ensure the post is structured with an "
            "engaging introduction, insightful body, "
            "and a summarizing conclusion.\n"
        "5. Proofread for grammatical errors and "
            "alignment with the brand's voice.\n"
    ),
    expected_output="A well-written blog post "
        "in markdown format, ready for publication, "
        "each section should have 2 or 3 paragraphs.",
    agent=writer,
)



### Task :- Edit


In [21]:
edit = Task(
    description=("Proofread the given blog post for "
                 "grammatical errors and "
                 "alignment with the brand's voice."),
    expected_output="A well-written blog post in markdown format, "
                    "ready for publication, "
                    "each section should have 2 or 3 paragraphs.",
    agent=editor
)



## Creating the Crew

- Create your crew of Agents
- Pass the tasks to be performed by those agents.
    - **Note**: *For this simple example*, the tasks will be performed sequentially (i.e they are dependent on each other), so the _order_ of the task in the list _matters_.
- `verbose=2` allows you to see all the logs of the execution.


In [22]:
crew = Crew(
    agents=[planner, writer, editor],
    tasks=[plan, write, edit],
    verbose=2
)

## Running the Crew

In [23]:
result = crew.kickoff(inputs={"topic": "Artificial Intelligence"})
from IPython.display import Markdown
Markdown(result)

 [DEBUG]: == Working Agent: Content Planner
 [INFO]: == Starting Task: 1. Prioritize the latest trends, key players, and noteworthy news on Artificial Intelligence.
2. Identify the target audience, considering their interests and pain points.
3. Develop a detailed content outline including an introduction, key points, and a call to action.
4. Include SEO keywords and relevant data or sources.


> Entering new CrewAgentExecutor chain...
I now can give a great answer

Final Answer:
Title: The Future of Artificial Intelligence: Trends, Players, and News

Introduction:
- Brief overview of the rapid advancements in artificial intelligence technology
- Mention of key players in the industry and recent noteworthy news

Key Points:
1. Latest Trends in Artificial Intelligence:
- Deep learning and neural networks
- Natural language processing
- Computer vision and image recognition
- AI in healthcare, finance, and other industries

2. Key Players in the AI Industry:
- Google (with Google AI and 

# The Future of Artificial Intelligence: Trends, Players, and News

## Introduction

Artificial intelligence (AI) technology has been rapidly advancing, shaping the way we interact with machines and data. Key players in the industry, such as Google, Amazon, IBM, Microsoft, and Facebook, continue to push the boundaries of AI innovation. Recent noteworthy news highlights breakthroughs in AI research, applications across various sectors, and ethical considerations in AI development.

## Latest Trends in Artificial Intelligence

### Deep Learning and Neural Networks
Deep learning algorithms and neural networks are at the forefront of AI innovation, enabling machines to learn from data and improve their performance over time. This technology powers applications such as virtual assistants, recommendation systems, and autonomous vehicles.

### Natural Language Processing
Natural language processing (NLP) allows machines to understand and generate human language, leading to advancements in speech recognition, machine translation, and sentiment analysis. NLP plays a crucial role in enhancing communication between humans and machines.

### Computer Vision and Image Recognition
Computer vision and image recognition technologies enable machines to interpret and analyze visual information, revolutionizing industries such as healthcare, retail, and security. These advancements have paved the way for facial recognition systems, object detection algorithms, and medical imaging tools.

### AI in Healthcare, Finance, and Other Industries
AI is transforming the healthcare and finance industries by streamlining processes, improving decision-making, and enhancing patient care. From personalized medicine to fraud detection, AI applications are reshaping the way these sectors operate, leading to improved outcomes and efficiency.

## Key Players in the AI Industry

### Google
Google is a major player in the AI industry, with initiatives such as Google AI and DeepMind driving research in machine learning, robotics, and natural language understanding. The company's AI-powered products and services are integrated into various platforms, enhancing user experiences and productivity.

### Amazon
Amazon's AWS AI services provide scalable and cost-effective solutions for businesses looking to implement AI technologies. From machine learning models to computer vision tools, Amazon's AI offerings cater to a wide range of industries, enabling companies to leverage data-driven insights for growth and innovation.

### IBM
IBM's Watson platform is a leader in cognitive computing and AI, offering solutions for industries such as healthcare, finance, and cybersecurity. With a focus on data analytics, machine learning, and natural language processing, IBM continues to drive innovation in AI research and development.

### Microsoft
Microsoft's Azure AI services empower organizations to build, deploy, and manage AI applications with ease. From speech recognition to predictive analytics, Microsoft's AI tools enable businesses to harness the power of data and drive intelligent decision-making across various domains.

### Facebook
Facebook's AI Research (FAIR) team is dedicated to advancing the field of AI through cutting-edge research and open-source collaborations. By focusing on projects such as computer vision, natural language understanding, and reinforcement learning, Facebook is at the forefront of AI innovation in social media and beyond.

## Noteworthy News in Artificial Intelligence

### Breakthroughs in AI Research
Recent breakthroughs in AI research include advancements in reinforcement learning, unsupervised learning, and generative models. These developments have led to improved performance in tasks such as image recognition, language translation, and autonomous navigation, pushing the boundaries of AI capabilities.

### Applications of AI in Various Sectors
AI is being applied across various sectors, from healthcare and finance to transportation and entertainment. Innovative use cases include personalized medicine, algorithmic trading, autonomous vehicles, and content recommendation systems, showcasing the diverse impact of AI technologies on society and business.

### Ethical Considerations and Regulations in AI Development
As AI technologies become more prevalent, ethical considerations and regulations are crucial in ensuring responsible AI development and deployment. Issues such as bias in algorithms, data privacy, and accountability in AI systems require careful attention and ethical frameworks to safeguard against potential risks and unintended consequences.

## Call to Action: Stay Informed

Stay informed about the latest AI trends, key players, and news by subscribing to our newsletter. Explore the exciting developments in artificial intelligence and discover how AI is shaping the future of technology and society.

By following this comprehensive content plan, we aim to provide our audience with valuable insights and information on the future of artificial intelligence. Join us on this journey of exploration and innovation in the world of AI.

*SEO Keywords: Artificial Intelligence trends, Key players in AI industry, Latest news in artificial intelligence, AI technology developments*

Resources:
1. "The State of Artificial Intelligence in 2021" - Forbes
2. "Top AI Companies Leading the Industry" - TechCrunch
3. "Ethical Considerations in AI Development" - Harvard Business Review